# Ocean sounds: Path A (128-band mel) diffusion

Trains the class-conditional mel-spectrogram diffusion model (Path A at 128 bands, the generator
approved for the distillation work) on the ocean sounds dataset: Nassau, yellowfin and black
grouper, squirrelfish, toadfish, and vessel sounds, recorded at 10 kHz in 2-second clips.

The model and training recipe are the ones used on ESC-50. What is new is the data handling,
driven by the dataset audit:

* **Duplicates.** The audit found exact duplicate clips. They are removed before splitting; a clip
  found under two different classes is dropped.
* **No recording IDs.** Filenames carry no recording, site or date. Consecutive file numbers are
  grouped into provisional "recordings" so neighbouring windows stay on one side of the split.
  **This is a stand-in until real source IDs are available.**
* **Very-low-frequency energy.** Toadfish clips are dominated by energy below 10 Hz, most likely DC
  offset or equipment rather than the call. Clips are high-passed at 20 Hz.
* **Level differences.** Classes differ by up to ~20 dB in raw level, possibly from equipment. Clips
  are normalized to a common RMS by default.
* **Class imbalance (7.7 : 1).** The generator is trained with class-balanced sampling, so the rare
  species are learned as well as the common ones.

Set `QUICK_TEST = True` first to run the whole notebook in a few minutes.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Audio, display

assert hasattr(C, "load_ocean"), "common.py is an old version: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Config

`SOURCE` is the dataset on Drive, either the zipped folder (fastest: it is extracted once to
Colab's local disk) or the unzipped folder. The data options change the cached dataset, so each
combination is cached separately.

In [ ]:
QUICK_TEST = False          # True: short run to check the notebook end to end

# data
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"   # zip or folder
GROUP_SIZE  = 20            # provisional recording groups: blocks of 20 consecutive file numbers
HIGHPASS_HZ = 20.0          # remove DC and sub-sonic drift; 0 to disable
NORMALIZE   = "rms"         # "rms": every clip to the same RMS level; "none": keep raw levels
BALANCED    = True          # class-balanced sampling for the generator

# model and sampling (Path A at 128 bands)
DROPOUT     = 0.1
GUIDANCE    = 1.0           # 1.0 for distribution metrics
N_PER_CLASS = 4 if QUICK_TEST else 100

OUT_ROOT = "/content/drive/MyDrive/thesis_runs/ocean"
cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data",
                     **(dict(total_steps=300, inference_steps=20) if QUICK_TEST else {}))
C.seed_everything(cfg.seed)

FE_TAG        = "" if (cfg.n_fft, cfg.f_max) == (1024, None) else f"_fft{cfg.n_fft}_fmax{cfg.f_max:g}"
TAG           = FE_TAG + f"_hp{HIGHPASS_HZ:g}_{NORMALIZE}" + ("_bal" if BALANCED else "") + ("_quick" if QUICK_TEST else "")
RUN_DIR       = f"{cfg.out_root}/path_a128{TAG}"
FRONTEND_FILE = f"{cfg.out_root}/frontend_128mel{FE_TAG}_hp{HIGHPASS_HZ:g}_{NORMALIZE}.json"
SAMPLES_FILE  = f"{cfg.out_root}/samples/path_a128{TAG}.pt"
SAMPLE_SHAPE  = (1, cfg.n_mels, cfg.spec_frames)
print(f"clip: {cfg.clip_samples} samples ({cfg.clip_seconds:.2f} s) -> {cfg.n_mels} mel bands x {cfg.spec_frames} frames, "
      f"{cfg.f_min:g} Hz to {cfg.sample_rate // 2} Hz")

## 3. Data

Loads every clip, removes duplicates, builds the provisional groups, assigns folds (1-3 train,
4 validation, 5 held out; about 20% of each class per fold, no group split across folds),
high-passes and normalizes. The first run reads ~50k files and takes a while; later runs load the
cache.

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
for k, v in data["report"].items():
    print(f"{k:<40}{v}")

names = data["class_names"]
print(f"\n{'class':<20}" + "".join(f"{'fold ' + str(f):>9}" for f in range(1, 6)) + f"{'total':>9}{'groups':>9}")
for li, n in enumerate(names):
    m = (data["labels"] == li)
    per_fold = [int(((data["folds"] == f) & m).sum()) for f in range(1, 6)]
    n_groups = len({g for g, keep in zip(data["groups"], m.tolist()) if keep})
    print(f"{n:<20}" + "".join(f"{x:>9}" for x in per_fold) + f"{int(m.sum()):>9}{n_groups:>9}")
    assert per_fold[cfg.val_fold - 1] > 0 and per_fold[cfg.heldout_fold - 1] > 0, f"{n} is missing from the validation or test fold"

train_split, val_split, heldout_split = C.split_by_fold(data, cfg)
report, class_names = data["report"], data["class_names"]
del data                                                     # the splits hold what is needed; frees ~2 GB
train_set = C.CropDataset(train_split, cfg)
train_data = C.BalancedDataset(train_set, train_split["labels"]) if BALANCED else train_set
print(f"\ntrain {len(train_split['labels'])} | val {len(val_split['labels'])} | held-out {len(heldout_split['labels'])} clips")

## 4. Mel front end

128 mel bands from 20 Hz to 5 kHz. The dB window is calibrated on the training split only and
saved, as for ESC-50. The plots and players below show one real clip per class, as a check that the
high-pass and normalization did not remove what matters.

In [ ]:
if os.path.exists(FRONTEND_FILE):
    frontend = C.MelFrontEnd.load(FRONTEND_FILE, cfg, device=device)
else:
    fit_idx = torch.randperm(len(train_split["labels"]))[:4000]
    frontend = C.MelFrontEnd(cfg, device=device).fit(train_split["waveforms"][fit_idx].float())
    frontend.save(FRONTEND_FILE)
print(f"dB window: [{frontend.db_max - cfg.db_range:.1f}, {frontend.db_max:.1f}]")

ref_wave, ref_labels = C.reference_crops(val_split, cfg, k=1)
fig, axes = plt.subplots(1, len(class_names), figsize=(3.2 * len(class_names), 3))
for li, ax in enumerate(axes):
    w = ref_wave[ref_labels == li][:1]
    ax.imshow(frontend(w)[0, 0].cpu(), origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
    ax.set_title(class_names[li], fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
for li, n in enumerate(class_names):
    print(n); display(Audio(ref_wave[ref_labels == li][0, 0].numpy(), rate=cfg.sample_rate))

## 5. Model

The Path A network, unchanged from the 128-band ESC-50 run, so results on the two datasets are
directly comparable. Each choice and the reason for it:

| Choice | Reason |
|---|---|
| Same UNet as Path A at 128 bands: three levels (64/128/256 channels), two ResNet blocks per level, attention only at the 32 x 32 bottleneck, scale-shift conditioning, ~16.4M parameters | It is the generator Ali approved for distillation, and the representation study chose it on ESC-50. Keeping it unchanged means the move to ocean data changes only the data, so differences in behaviour can be attributed to the domain. |
| Class embedding with 6 classes + a "no label" entry, label dropout 10% | Same conditioning as on ESC-50; the "no label" entry keeps classifier-free guidance available. |
| 128 mel bands from 20 Hz to 5 kHz | At 10 kHz the whole band up to Nyquist is kept. The mel scale gives the low frequencies, where grouper, toadfish and squirrelfish calls sit, the most bands; 128 bands beat 64 on ESC-50 under both embedders. The 20 Hz lower edge matches the high-pass, so no band is spent on removed content. |
| 1,024-point FFT (102 ms window, 9.8 Hz bins) and a 156-sample hop (15.6 ms) | The window resolves the low-frequency tonal calls to under 10 Hz while keeping pulses and clicks visible in time; the hop gives exactly 128 frames for a 1.98 s crop, the same 128 x 128 input as on ESC-50. |
| High-pass at 20 Hz before anything else | The audit found toadfish energy concentrated below 10 Hz and ~20 dB louder than other classes, most likely DC offset or equipment. Left in, both the generator and any classifier could tell classes apart by recorder instead of by sound. |
| Per-clip RMS normalization | Removes level differences that may come from gain or distance rather than the species. Kept as an option (`NORMALIZE = "none"`) because absolute level can carry information; worth confirming with Ali. |
| Class-balanced sampling | With 7.7 : 1 imbalance, uniform sampling would show the model toadfish 7x more often than black grouper. Balanced draws give every class the same training signal, which matters because the rare classes are exactly where synthetic data is most useful. |
| v-prediction, cosine schedule, EMA, 10,000 steps, batch 32 | Same recipe as every ESC-50 path. 10k steps x 32 = 320k draws, about 11 passes over the ~30k training clips. The dataset is ~25x larger than ESC-50, so longer training may help later, but the first run stays comparable. |

In [ ]:
def forward_fn(model, x, t, labels):
    return model(x, t, class_labels=labels).sample

unet = C.build_unet2d(cfg, dropout=DROPOUT).to(device)
n_params = sum(p.numel() for p in unet.parameters())
print(f"params: {n_params:,}")

## 6. Train

Resumes from `RUN_DIR/last.pt` if it exists.

In [ ]:
t0 = time.time()
ema, losses = C.train_diffusion(unet, forward_fn, frontend, train_data, cfg, RUN_DIR, device)
print(f"training time this session: {(time.time() - t0) / 60:.1f} min")
plt.figure(figsize=(5, 3))
plt.plot([cfg.log_every * (i + 1) for i in range(len(losses))], losses)
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Ocean Path A training loss"); plt.show()

### Learnability

The same check as on ESC-50: v-prediction loss relative to the best "learned nothing" predictor,
per mel band. 1.0 means nothing was learned; Path A on ESC-50 scored 0.121 at 128 bands.

In [ ]:
scheduler, _ = C.make_schedulers(cfg)
ema.store(unet.parameters()); ema.copy_to(unet.parameters()); unet.eval()
batch = [train_data[i] for i in range(128)]
x0 = frontend(torch.stack([b[0] for b in batch]).to(device))
labels = torch.stack([b[1] for b in batch]).to(device)
var = x0.var(dim=(0, 1, 3)).cpu()
err, base = torch.zeros(cfg.n_mels), torch.zeros(cfg.n_mels)
with torch.no_grad():
    for t_val in (100, 300, 500, 700, 900):
        t = torch.full((len(x0),), t_val, device=device)
        noise = torch.randn_like(x0)
        pred = forward_fn(unet, scheduler.add_noise(x0, noise, t), t, labels)
        err += ((pred - scheduler.get_velocity(x0, noise, t)) ** 2).mean(dim=(0, 1, 3)).cpu() / 5
        ac = scheduler.alphas_cumprod[t_val].item(); a, b = ac ** 0.5, (1 - ac) ** 0.5
        base += ((a**2 + b**2 * var) - (a * b * (1 - var)) ** 2 / (a**2 * var + b**2)) / 5
ema.restore(unet.parameters()); unet.train()
rel = err / base.clamp(min=1e-8)
print(f"variance-weighted relative v-loss: {(rel * var).sum() / var.sum():.3f}  (ESC-50, 128 bands: 0.121)")

## 7. Generate and save samples

A class-balanced set (`N_PER_CLASS` per class), sampled from the EMA weights.

In [ ]:
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()
gen_labels = C.balanced_labels(cfg, N_PER_CLASS)
generated = C.sample(unet, forward_fn, SAMPLE_SHAPE, cfg, device, labels=gen_labels, ema=ema, guidance=GUIDANCE)
meta = dict(path="A128_ocean", dataset="ocean", guidance=GUIDANCE, steps=cfg.inference_steps, n_params=n_params,
            train_steps=cfg.total_steps, balanced=BALANCED, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE,
            group_size=GROUP_SIZE, data_report=report, class_names=class_names, commit=commit)
C.save_samples(SAMPLES_FILE, generated, gen_labels, cfg, representation="mel", **meta)
print("saved", tuple(generated.shape), "->", SAMPLES_FILE)

## 8. Look and listen

Real validation clips (top) against generated ones of the same class (bottom), then audio: real,
real through mel + Griffin-Lim (the representation's ceiling), and two generated clips per class.
Judge the generated audio against the Griffin-Lim version, not the original.

In [ ]:
fig, axes = plt.subplots(2, len(class_names), figsize=(3.2 * len(class_names), 5.5))
for li in range(len(class_names)):
    real = frontend(ref_wave[ref_labels == li][:1])[0, 0].cpu()
    fake = generated[gen_labels == li][0, 0]
    for row, (img, tag) in enumerate([(real, "real"), (fake, "generated")]):
        axes[row, li].imshow(img, origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        axes[row, li].set_title(f"{class_names[li]} ({tag})", fontsize=8); axes[row, li].set_xticks([]); axes[row, li].set_yticks([])
plt.tight_layout(); plt.show()

for li, n in enumerate(class_names):
    real = ref_wave[ref_labels == li][:1]
    real_gl = C.spec_to_wave(frontend(real).cpu(), frontend, n_iter=64)
    gen_gl = C.spec_to_wave(generated[gen_labels == li][:2], frontend, n_iter=64)
    print(f"{n}: real, real via mel + Griffin-Lim, then generated")
    for w in [real[0, 0], real_gl[0, 0]] + [g[0] for g in gen_gl]:
        display(Audio(w.numpy(), rate=cfg.sample_rate))